# Conversational and Multi-Turn RAG

Single-turn RAG (**02**) treats each question in isolation. Real users **follow up**: *"What command do I run?"* after discussing Alembic, or *"Can you elaborate on step 2?"* after a migration answer. **Conversational RAG** must carry **dialogue history**, resolve **coreference** ("it", "that"), and often **re-retrieve** with a **standalone query** derived from context.

This notebook covers multi-turn challenges, architecture patterns, **query condensation**, memory strategies (buffer, summary, entity), when to re-retrieve vs reuse chunks, prompt layout per turn, token budgeting across history + context (**06**), session state in production (**10**), and multi-turn eval scripts.

Prerequisites: **10. Production Patterns for RAG**, **06. Context Assembly**, **05. Prompting**, **02.07 Tokens and Context Windows**. Next: **12. RAG vs Agents and Tool Use**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json
from dataclasses import dataclass, field

import chromadb
import numpy as np
import matplotlib.pyplot as plt
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

DOCUMENTS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI and stores notes in memory for demos. Routes live under /notes.", "source": "api-docs"},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling new revisions.", "source": "db-docs"},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Clients send Authorization: Bearer token header.", "source": "security-docs"},
    {"id": "c4", "text": "Pytest fixtures share database setup across tests. Use conftest.py for session-scoped engines.", "source": "test-docs"},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live database schema and drafts revision files.", "source": "db-docs"},
]

print("Setup OK")

---

## 1. Single-Turn vs Multi-Turn

| Dimension | Single-turn (**02**) | Multi-turn (this notebook) |
|-----------|----------------------|----------------------------|
| Query to embed | User message verbatim | **Condensed standalone** query |
| Context in prompt | Retrieved chunks | Chunks + **trimmed history** |
| Retrieval | Once per request | Often **per turn** |
| Failure mode | Wrong chunk once | **Stale chunks** on follow-up |
| Eval | Per-question golden | **Conversation scripts** (**09**) |
| Token budget | System + chunks + Q | System + chunks + **history** + Q |

Multi-turn RAG is still RAG — not an agent loop (**12**). The LLM does not choose tools; you manage **memory** and **when to search again**.

---

## 2. Multi-Turn Challenges

| Challenge | Example | Risk |
|-----------|---------|------|
| **Coreference** | "How do I run it?" after Alembic talk | Embed "it" → wrong chunks |
| **Topic drift** | JWT question after migration thread | Stale c2 chunks in context |
| **Ellipsis** | "And for pytest?" | Missing subject in embed query |
| **Window growth** | 20-turn support chat | History eats chunk budget (**06**) |
| **Compound hallucination** | Turn 1 wrong → Turn 2 builds on it | Trusting assistant text as facts |

```
Turn 1: "Tell me about Alembic"     → retrieve c2, c5
Turn 2: "How do I run it?"          → MUST NOT embed raw follow-up alone
Turn 3: "How does JWT auth work?"   → MUST re-retrieve c3, drop Alembic bias
```

---

## 3. Conversational RAG Architecture

Typical flow for turn $N$:

```
Session history ──► Query condenser ──► Retrieve(standalone_q)
                           │                    │
                           ▼                    ▼
                    Topic change?          Fresh chunks
                           │                    │
                           └──────► Assemble(history + chunks) ──► LLM
```

### 3.1 Retrieval Strategies per Turn

| Variant | Re-retrieve? | Best for |
|---------|--------------|----------|
| **Always re-retrieve** | Every turn | Shifting topics, production default |
| **Cache turn-1 chunks** | No | Deep dive on same doc only |
| **Hybrid** | If topic-change detector fires | Cost/latency optimization |

**Default recommendation:** re-retrieve every turn with a **condensed standalone query** — simpler and safer than chunk caching.

---

## 4. Standalone Query Generation (Query Condensation)

Transform `history + follow_up` into a **search-ready** question before embedding.

```
History:  User: How does Alembic handle migrations?
          Assistant: Alembic applies SQLAlchemy schema migrations...
Follow-up: How do I run it?

Standalone: What command applies Alembic migrations to the latest revision?
```

| Method | Pros | Cons |
|--------|------|------|
| **LLM condenser** | Handles complex coreference | Extra API call |
| **Rule-based** | Free, fast | Brittle |
| **Last-N-turns + follow-up concat** | Simple baseline | Noisy embedding |

In [ ]:
def condense_rule_based(history: list[tuple[str, str]], follow_up: str) -> str:
    """Illustrative rule-based condenser for teaching."""
    recent_user = " ".join(u for u, _ in history[-3:]).lower()
    fu = follow_up.lower()
    if "alembic" in recent_user or "migration" in recent_user:
        if any(w in fu for w in ["run", "command", "it", "execute"]):
            return "What command applies Alembic migrations with alembic upgrade head?"
    if "jwt" in recent_user or "auth" in recent_user:
        return f"JWT API authentication: {follow_up}"
    if "pytest" in recent_user or "test" in recent_user:
        return f"Pytest fixtures conftest: {follow_up}"
    return follow_up


history = [("How does Alembic handle database migrations?", "Alembic applies SQLAlchemy schema migrations.")]
for follow_up in ["How do I run it?", "What about autogenerate?"]:
    print(f"Follow-up: {follow_up!r}")
    print(f"  Standalone: {condense_rule_based(history, follow_up)!r}\n")

In [ ]:
CONDENSE_SYSTEM = """Rewrite the follow-up question as a standalone search query.
Use the chat history to resolve pronouns (it, that, this).
Output ONLY the standalone question — no explanation."""


def condense_llm(history: list[tuple[str, str]], follow_up: str) -> str:
    lines = []
    for user, assistant in history[-4:]:
        lines.append(f"User: {user}")
        lines.append(f"Assistant: {assistant}")
    transcript = "\n".join(lines)
    r = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": CONDENSE_SYSTEM},
            {"role": "user", "content": f"History:\n{transcript}\n\nFollow-up: {follow_up}"},
        ],
    )
    return r.choices[0].message.content.strip()


# Uncomment to run live:
# print(condense_llm(history, "How do I run it?"))

---

## 5. Memory Strategies

| Memory type | Stores | Use when |
|-------------|--------|----------|
| **Buffer** | Full transcript | Short chats (< 6 turns) |
| **Summary** | LLM-compressed older turns | Long sessions |
| **Entity** | Extracted facts (user, env) | Support tickets |
| **Retrieval cache** | Last turn's chunk ids | Same-topic deep dive only |

### 5.1 What NOT to Store as Ground Truth

Do **not** treat previous **assistant answers** as authoritative documents. They may contain hallucinations. For facts, always prefer **fresh retrieval** from the KB over replaying assistant prose.

---

## 6. Topic Change Detection

Decide whether to **force re-retrieve** or invalidate cached chunks.

| Signal | Action |
|--------|--------|
| Embedding distance(condensed_q, last_q) > threshold | New topic → full re-retrieve |
| User clicks "New topic" | Clear history cache |
| Keyword shift (jwt vs alembic) | Re-retrieve |
| Same condensed query as prior turn | Optional chunk reuse |

In [ ]:
def topic_shift_heuristic(last_topic: str, new_query: str) -> bool:
    topics = {
        "alembic": ["alembic", "migration", "schema", "upgrade"],
        "auth": ["jwt", "bearer", "auth", "token"],
        "test": ["pytest", "fixture", "conftest", "test"],
    }
    nq = new_query.lower()
    for name, kws in topics.items():
        if name == last_topic:
            continue
        if any(k in nq for k in kws):
            return True
    return False


print("alembic → jwt:", topic_shift_heuristic("alembic", "How does JWT authentication work?"))
print("alembic → run it:", topic_shift_heuristic("alembic", "How do I run the upgrade command?"))

---

## 7. Prompt Layout for Turn k

```
system:  grounding rules (**05**)
user:    [Optional: summary of turns 1..k-3]
         Context (retrieved THIS turn):
           <doc id="c2">...</doc>
         Recent dialogue (last 1–2 exchanges):
           User: ...
           Assistant: ...
         Current question: {follow_up}
```

| Priority if over budget (**06**) | Drop order |
|----------------------------------|------------|
| 1 (keep) | System grounding rules |
| 2 | Fresh retrieved chunks |
| 3 | Current question |
| 4 | Recent dialogue (last 2 turns) |
| 5 (drop first) | Older history / summary |

---

## 8. Token Budget Across Turns

$$T_{\text{history}} + T_{\text{context}} + T_{\text{question}} + T_{\text{system}} \leq C_{\max} - T_{\text{reserved\_output}}$$

History grows every turn — trim **oldest** segments first, then reduce chunk count (**06**).

In [ ]:
BUDGET = 4000
SYSTEM_TOKENS = 220
OUTPUT_RESERVE = 512
CONTEXT_TOKENS = 1200
QUESTION_TOKENS = 80
history_segments = [350, 420, 280, 190, 310]  # growing transcript

available = BUDGET - SYSTEM_TOKENS - OUTPUT_RESERVE - CONTEXT_TOKENS - QUESTION_TOKENS
kept = list(history_segments)
while sum(kept) > available and kept:
    kept.pop(0)

print(f"History budget: {available} tokens")
print(f"Original segments: {len(history_segments)} ({sum(history_segments)} tok)")
print(f"Kept segments:     {len(kept)} ({sum(kept)} tok)")

In [ ]:
turns = list(range(1, len(history_segments) + 1))
cumulative = np.cumsum(history_segments)
plt.figure(figsize=(7, 4))
plt.plot(turns, cumulative, marker="o", linewidth=2, label="History tokens")
plt.axhline(available, color="red", linestyle="--", label="History budget")
plt.xlabel("Conversation turn")
plt.ylabel("Cumulative history tokens")
plt.title("History grows — plan trimming or summarization")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 9. Session State in Production

Store per `session_id` (**10**):

| Field | Purpose |
|-------|--------|
| `messages` | Full or summarized transcript |
| `last_condensed_query` | Topic shift detection |
| `last_retrieved_ids` | Debug + optional cache |
| `index_version` | Invalidate on deploy |
| `created_at` | Session TTL |

Use Redis or DB — not in-process dicts — when running multiple API workers.

In [ ]:
@dataclass
class ChatSession:
    session_id: str
    index_version: str
    messages: list[dict] = field(default_factory=list)
    last_condensed_query: str = ""
    last_retrieved_ids: list[str] = field(default_factory=list)

    def add_turn(self, user: str, assistant: str) -> None:
        self.messages.append({"role": "user", "content": user})
        self.messages.append({"role": "assistant", "content": assistant})

    def history_pairs(self) -> list[tuple[str, str]]:
        pairs = []
        i = 0
        while i + 1 < len(self.messages):
            if self.messages[i]["role"] == "user":
                pairs.append((self.messages[i]["content"], self.messages[i + 1]["content"]))
            i += 2
        return pairs


session = ChatSession(session_id="sess-001", index_version="notes_api_v1")
session.add_turn("How does Alembic work?", "Alembic manages SQLAlchemy migrations.")
print("History pairs:", len(session.history_pairs()))

---

## 10. Hands-On — Multi-Turn RAG Pipeline

Index corpus, condense follow-up, retrieve with standalone query, generate answer.

In [ ]:
def embed_texts(texts: list[str]) -> list[list[float]]:
    r = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in sorted(r.data, key=lambda x: x.index)]


client = chromadb.Client()
try:
    client.delete_collection("chat_rag")
except Exception:
    pass
col = client.create_collection("chat_rag", metadata={"hnsw:space": "cosine"})
for d in DOCUMENTS:
    col.add(ids=[d["id"]], documents=[d["text"]], embeddings=embed_texts([d["text"]]),
            metadatas=[{"source": d["source"]}])


def retrieve(query: str, k: int = 3) -> list[str]:
    r = col.query(query_embeddings=[embed_texts([query])[0]], n_results=k)
    return r["ids"][0]


RAG_SYSTEM = "Answer ONLY from Context. End with Sources: <chunk ids>"


def chat_rag_turn(session: ChatSession, follow_up: str, k: int = 3) -> tuple[str, list[str]]:
    standalone = condense_rule_based(session.history_pairs(), follow_up)
    session.last_condensed_query = standalone
    chunk_ids = retrieve(standalone, k=k)
    session.last_retrieved_ids = chunk_ids
    got = col.get(ids=chunk_ids, include=["documents"])
    ctx = "\n\n".join(f"[{cid}] {doc}" for cid, doc in zip(chunk_ids, got["documents"]))
    history_text = "\n".join(f"User: {u}\nAssistant: {a}" for u, a in session.history_pairs()[-2:])
    user_msg = f"Context:\n{ctx}\n\nRecent dialogue:\n{history_text}\n\nCurrent question: {follow_up}"
    r = openai_client.chat.completions.create(
        model=CHAT_MODEL, temperature=0,
        messages=[{"role": "system", "content": RAG_SYSTEM}, {"role": "user", "content": user_msg}],
    )
    answer = r.choices[0].message.content
    session.add_turn(follow_up, answer)
    return answer, chunk_ids

In [ ]:
# Simulated 3-turn conversation (run with API key for live LLM)
demo = ChatSession(session_id="demo", index_version="v1")
demo.add_turn("Tell me about Alembic migrations.", "Alembic applies SQLAlchemy schema migrations.")

turns = [
    "How do I run it?",
    "How does JWT authentication work for the API?",
]

for t in turns:
    standalone = condense_rule_based(demo.history_pairs(), t)
    ids = retrieve(standalone, k=3)
    print(f"\nFollow-up: {t}")
    print(f"  Standalone: {standalone}")
    print(f"  Retrieved:  {ids}")
    # answer, ids = chat_rag_turn(demo, t)  # uncomment for full LLM

Turn 2 should still retrieve **c2** (Alembic). Turn 3 should shift to **c3** (JWT) — not reuse migration chunks.

---

## 11. Evaluating Multi-Turn RAG

Extend golden set (**09**) with **conversation scripts**:

| Turn | User | Expected retrieval | Expected behavior |
|------|------|-------------------|-------------------|
| 1 | "Tell me about Alembic" | c2, c5 | Overview answer |
| 2 | "How do I run it?" | c2 | Mentions `upgrade head` |
| 3 | "What about JWT auth?" | c3 | Topic shift; not c2 |
| 4 | "Does API use GraphQL?" | none | Refusal (**08**) |

Metrics: **per-turn Recall@k**, condensation quality (standalone hits gold), refusal on turn 4.

In [ ]:
CONVERSATION_EVAL = [
    {"turn": 1, "user": "Tell me about Alembic migrations", "gold_ids": ["c2", "c5"]},
    {"turn": 2, "user": "How do I run it?", "gold_ids": ["c2"], "history": 1},
    {"turn": 3, "user": "How does JWT auth work?", "gold_ids": ["c3"], "history": 2},
]

sim_history: list[tuple[str, str]] = []
for step in CONVERSATION_EVAL:
    standalone = condense_rule_based(sim_history, step["user"]) if step.get("history") else step["user"]
    retrieved = retrieve(standalone, k=3)
    hit = bool(set(step["gold_ids"]) & set(retrieved))
    print(f"Turn {step['turn']}: hit={hit}  retrieved={retrieved}  standalone={standalone[:50]}...")
    sim_history.append((step["user"], "(assistant placeholder)"))

---

## 12. UX and Latency Considerations

| Pattern | Why |
|---------|-----|
| "Searching docs…" on re-retrieve | Sets latency expectation (2× LLM if condenser) |
| Sources **per turn** | User verifies each answer |
| **New topic** button | Forces history bias reset |
| Show condensed query (debug mode) | Support troubleshooting |
| Session TTL (e.g. 30 min) | Bound memory and cost |

---

## 13. Anti-Patterns

| Anti-pattern | Why it fails |
|--------------|-------------|
| Embed raw follow-up ("How do I run it?") | Coreference miss |
| Never re-retrieving | JWT question gets Alembic chunks |
| Stuff entire chat into embed query | Diluted semantic signal |
| Trust assistant paraphrases as KB | Compound hallucination |
| Unbounded session length | Token blow-up + cost |
| Skip condensation to save 1 LLM call | Retrieval miss costs more |

---

## 14. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Same k and prompt as single-turn | Ignores history budget | Trim + condense |
| Store session in memory only | Lost on restart / wrong worker | Redis session store |
| No index_version on session | Stale answers after deploy | **10** manifest |
| Eval only single-turn | Multi-turn regressions ship | Conversation scripts |
| Summary memory without eval | Loses critical details | Test summary quality |

---

## 15. Summary

Conversational RAG extends single-turn pipelines with **query condensation**, **session memory**, **per-turn retrieval**, and **history-aware token budgeting**. Condense follow-ups into standalone search queries before embedding; re-retrieve when topics shift; never treat past assistant answers as ground truth.

Demonstrations covered rule-based and LLM condensation, topic-shift heuristics, history trimming, `ChatSession` state, multi-turn retrieval eval, and a `chat_rag_turn` pipeline on **c1–c5**.

Back: **10. Production Patterns for RAG**. Next: **12. RAG vs Agents and Tool Use** — when retrieval is enough and when to add tools.